<a href="https://colab.research.google.com/github/sowmya236/demo-123/blob/master/SMS_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

urllib.request.urlretrieve(url, "sms_dataset.zip")

with zipfile.ZipFile("sms_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("sms_data")

file_path = "sms_data/SMSSpamCollection"

data = pd.read_csv(
    file_path,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8"
)

print("Dataset downloaded successfully!")
print(data.head())
print("\nDataset shape:", data.shape)
print("\nLabel counts:")
print(data["label"].value_counts())


Dataset downloaded successfully!
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset shape: (5572, 2)

Label counts:
label
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import joblib

# Remove duplicate messages
data = data.drop_duplicates(subset="message")

# Input and output
X = data["message"]
y = data["label"]

# Split dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Create the ML pipeline
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )),
    ("classifier", MultinomialNB())
])

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Save the complete model
joblib.dump(model, "sms_fraud_model.pkl")

print("\nModel saved successfully!")


Model Accuracy: 95.36%

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       903
        spam       1.00      0.63      0.78       131

    accuracy                           0.95      1034
   macro avg       0.97      0.82      0.87      1034
weighted avg       0.96      0.95      0.95      1034


Confusion Matrix:
[[903   0]
 [ 48  83]]

Model saved successfully!


In [ ]:

test_messages = [
    "Congratulations! You won a cash prize. Click here to claim now.",
    "Hi, are you coming to college tomorrow?",
    "URGENT! Claim your free reward now by clicking this link.",
    "Your class starts at 9 AM. Please bring your notebook."
]

for message in test_messages:
    prediction = model.predict([message])[0]

    if prediction == "spam":
        result = "SPAM / POSSIBLE FRAUD"
    else:
        result = "LEGITIMATE / NORMAL"

    print("Message:", message)
    print("Prediction:", result)
    print("-" * 60)


Message: Congratulations! You won a cash prize. Click here to claim now.
Prediction: SPAM / POSSIBLE FRAUD
------------------------------------------------------------
Message: Hi, are you coming to college tomorrow?
Prediction: LEGITIMATE / NORMAL
------------------------------------------------------------
Message: URGENT! Claim your free reward now by clicking this link.
Prediction: SPAM / POSSIBLE FRAUD
------------------------------------------------------------
Message: Your class starts at 9 AM. Please bring your notebook.
Prediction: LEGITIMATE / NORMAL
------------------------------------------------------------


In [ ]:

%%writefile app.py

import streamlit as st
import joblib

# Load the trained model
model = joblib.load("sms_fraud_model.pkl")

st.set_page_config(
    page_title="SMS Fraud Detection",
    page_icon="📱",
    layout="centered"
)

st.title("📱 SMS Fraud Message Detection")
st.write(
    "Enter an SMS message to check whether it may be spam "
    "or a normal message."
)

message = st.text_area(
    "Enter your SMS message:",
    placeholder="Example: Congratulations! You won a prize..."
)

if st.button("Detect Message", type="primary"):
    if not message.strip():
        st.warning("Please enter an SMS message.")
    else:
        prediction = model.predict([message])[0]

        st.subheader("Prediction Result")

        if prediction == "spam":
            st.error("⚠️ SPAM / POSSIBLE FRAUD")
            st.write(
                "This message resembles spam. Avoid clicking "
                "suspicious links or sharing personal details."
            )
        else:
            st.success("✅ LEGITIMATE / NORMAL")
            st.write(
                "The model classified this message as normal. "
                "This does not guarantee that it is safe."
            )

st.divider()
st.caption(
    "Educational project. AI predictions can be incorrect. "
    "Never share OTPs, passwords, or banking PINs."
)


Writing app.py


In [ ]:

%%writefile requirements.txt
streamlit
scikit-learn
joblib


Writing requirements.txt


In [ ]:

from google.colab import files

files.download("app.py")
files.download("sms_fraud_model.pkl")
files.download("requirements.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

%%writefile requirements.txt
streamlit
scikit-learn
joblib


Overwriting requirements.txt


In [ ]:

from google.colab import files
files.download("requirements.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>